In [15]:
from pathlib import Path
import xml.etree.ElementTree as ET
import pandas as pd

In [18]:
xml_file = Path(r"C:\Users\xuzhao\Downloads\xml test\VARightsExport_18.1_20260828.xml")
tree = ET.parse(xml_file)
root = tree.getroot()

In [19]:
for element in root.iter():
    print(element.tag, element.attrib, element.text)

privileges {} 
  
privilegesversion {} 36
privilege {} 
    
privilegeid {} QASignOffTestResult
privilegeversion {} 1
privilegecategory {} Treatment Delivery
privilegegroup {} Quality Assurance
privilegedisplayname {} Change QA Test Results Status
usergroups {} 
      
groupcuid {'allow': 'false'} SysAdmin
groupcuid {'allow': 'false'} PIClerical
groupcuid {'allow': 'false'} Service
groupcuid {'allow': 'false'} Oncologist
groupcuid {'allow': 'false'} mdt
groupcuid {'allow': 'false'} Physics
groupcuid {'allow': 'false'} HCA_Physics
groupcuid {'allow': 'false'} Level3
groupcuid {'allow': 'false'} Remote
groupcuid {'allow': 'false'} SpecialistRads
groupcuid {'allow': 'false'} ScriptAdvPhys
groupcuid {'allow': 'false'} AllRights
groupcuid {'allow': 'false'} CLERICAL
groupcuid {'allow': 'false'} Varian_PBT
groupcuid {'allow': 'false'} Level1
groupcuid {'allow': 'false'} Nurse
groupcuid {'allow': 'false'} Level2
groupcuid {'allow': 'false'} PBT_Physics
groupcuid {'allow': 'false'} AdvTherapis

In [20]:
rows = []

for privilege in root.findall("privilege"):
    row = {}

    row["privilegeid"] = privilege.findtext("privilegeid")
    row["privilegeversion"] = privilege.findtext("privilegeversion")
    row["privilegecategory"] = privilege.findtext("privilegecategory")
    row["privilegegroup"] = privilege.findtext("privilegegroup")
    row["privilegedisplayname"] = privilege.findtext("privilegedisplayname")

    for group in privilege.findall("usergroups/groupcuid"):
        group_name = group.text
        allow_value = group.get("allow")

        row[group_name] = allow_value

    rows.append(row)

df = pd.DataFrame(rows)

df

,privilegeid,privilegeversion,privilegecategory,privilegegroup,privilegedisplayname,SysAdmin,PIClerical,Service,Oncologist,mdt,...,CLERICAL,Varian_PBT,Level1,Nurse,Level2,PBT_Physics,AdvTherapist,Level0,QualityTeam,AdvPhysics
0,QASignOffTestResult,1,Treatment Delivery,Quality Assurance,Change QA Test Results Status,false,false,false,false,false,...,false,false,false,false,false,false,false,false,false,false
1,Delete_Completed_Activities,3,Practice Management,Charges/Activity,Delete Completed Activities,true,false,true,false,false,...,false,true,false,false,false,false,true,false,false,false
2,DeleteTxBeam,2,Treatment Delivery,Plan Modification,Delete Treatment Field,false,false,true,false,false,...,false,false,false,false,false,false,true,false,false,false
3,Read_Access_to_Quality_Measures_Area,2,Practice Management,Patient Management,Read Access to Quality Measures,false,false,false,false,false,...,false,false,false,true,false,false,false,false,false,false
4,Full_Access_to_InVivo_Dosimetry,1,Practice Management,Treatment History,Full Access to In Vivo Dosimetry,true,false,false,false,false,...,false,true,false,false,false,false,false,false,false,false
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
332,ImageReferenceModify,2,Image review,Reference Images,Edit/Align/Assign Reference Images,true,false,false,true,false,...,false,true,true,false,true,true,true,true,false,true
333,HighDoseTreatment,2,Treatment Delivery,Safety,High Dose Treatment,true,false,true,false,false,...,false,true,false,false,false,true,true,false,false,true
334,Edit_CBCTp_Protocols,1,Treatment Delivery,Treatment,Edit CBCTp Protocols,false,false,false,false,false,...,false,false,false,false,false,false,false,false,false,false
335,Full_Access_to_Time_Planner,4,Practice Management,Patient Management,Full Access to Appointment Scheduling,true,true,true,true,false,...,true,true,true,true,true,true,true,true,true,true


In [21]:
# Sort rows by privilege ID
df = df.sort_values(by="privilegeid")

fixed_cols = [
    "privilegeid",
    "privilegeversion",
    "privilegecategory",
    "privilegegroup",
    "privilegedisplayname"
]

group_cols = sorted(
    [col for col in df.columns if col not in fixed_cols]
)

df = df[fixed_cols + group_cols]

output_file = xml_file.with_suffix(".xlsx")

df.to_excel(output_file, index=False)